In [1]:
import numpy as np
import pandas as pd

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

In [85]:
df=pd.read_csv('Personal_Finance_Dataset.csv')

In [86]:
df.head()

,Date,Transaction Description,Category,Amount,Type
0,2020-01-02,Score each.,Food & Drink,1485.69,Expense
1,2020-01-02,Quality throughout.,Utilities,1475.58,Expense
2,2020-01-04,Instead ahead despite measure ago.,Rent,1185.08,Expense
3,2020-01-05,Information last everything thank serve.,Investment,2291.00,Income
4,2020-01-13,Future choice whatever from.,Food & Drink,1126.88,Expense


In [87]:
df.columns

Index(['Date', 'Transaction Description', 'Category', 'Amount', 'Type'], dtype='object')

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 5 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Date                     1500 non-null   object 
 1   Transaction Description  1500 non-null   object 
 2   Category                 1500 non-null   object 
 3   Amount                   1500 non-null   float64
 4   Type                     1500 non-null   object 
dtypes: float64(1), object(4)
memory usage: 58.7+ KB


In [88]:
df.isna().sum()

Date                       0
Transaction Description    0
Category                   0
Amount                     0
Type                       0
dtype: int64

In [89]:
df['Date']=pd.to_datetime(df['Date'])

In [90]:
df['Month']=df['Date'].dt.to_period("M")

In [91]:
expenses=df[df['Type']=='Expense']

In [11]:
expenses.head()

,Date,Transaction Description,Category,Amount,Type,Month
0,2020-01-02,Score each.,Food & Drink,1485.69,Expense,2020-01
1,2020-01-02,Quality throughout.,Utilities,1475.58,Expense,2020-01
2,2020-01-04,Instead ahead despite measure ago.,Rent,1185.08,Expense,2020-01
4,2020-01-13,Future choice whatever from.,Food & Drink,1126.88,Expense,2020-01
5,2020-01-14,Benefit suggest page southern.,Shopping,448.68,Expense,2020-01


In [92]:
expenses.drop(columns=['Transaction Description'],inplace=True)

C:\Users\Sriza Goel\AppData\Local\Temp\ipykernel_34452\244276403.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  expenses.drop(columns=['Transaction Description'],inplace=True)


In [93]:
expenses.head()

,Date,Category,Amount,Type,Month
0,2020-01-02,Food & Drink,1485.69,Expense,2020-01
1,2020-01-02,Utilities,1475.58,Expense,2020-01
2,2020-01-04,Rent,1185.08,Expense,2020-01
4,2020-01-13,Food & Drink,1126.88,Expense,2020-01
5,2020-01-14,Shopping,448.68,Expense,2020-01


In [94]:
monthly=expenses.pivot_table(
    index="Month",
    columns="Category",
    values="Amount",
    aggfunc="sum"
)

In [95]:
monthly.head(15)

Category,Entertainment,Food & Drink,Health & Fitness,Rent,Salary,Shopping,Travel,Utilities
Month,,,,,,,,
2020-01,1914.85,5458.51,2370.34,3663.22,1077.09,1178.66,NaN,1475.58
2020-02,3199.52,1829.57,766.81,1222.17,294.31,1499.79,7272.28,1023.96
2020-03,1887.05,1353.18,881.82,6241.21,NaN,1929.08,486.51,802.96
2020-04,3821.84,1938.30,1603.18,NaN,2222.60,388.92,3195.04,3063.17
2020-05,NaN,2501.62,2893.98,3822.55,318.05,1318.17,4288.25,1703.51
2020-06,4092.44,426.30,4901.31,1547.80,1107.84,2611.18,990.97,5002.88
2020-07,4853.64,1786.39,3031.75,3919.73,109.93,2158.36,5355.49,2810.98
2020-08,2473.45,5614.45,2812.65,2853.72,1409.98,6623.78,6221.36,3377.41
2020-09,393.15,2596.78,214.33,5778.35,5906.79,799.34,4846.19,980.22


In [96]:
monthly.isna().sum()

Category
Entertainment       4
Food & Drink        5
Health & Fitness    4
Rent                6
Salary              3
Shopping            5
Travel              6
Utilities           4
dtype: int64

In [97]:
monthly=monthly.fillna(0)

In [98]:
monthly.isna().sum()

Category
Entertainment       0
Food & Drink        0
Health & Fitness    0
Rent                0
Salary              0
Shopping            0
Travel              0
Utilities           0
dtype: int64

In [99]:
monthly['total_spending']=monthly.sum(axis=1)

In [100]:
monthly.head()

Category,Entertainment,Food & Drink,Health & Fitness,Rent,Salary,Shopping,Travel,Utilities,total_spending
Month,,,,,,,,,
2020-01,1914.85,5458.51,2370.34,3663.22,1077.09,1178.66,0.00,1475.58,17138.25
2020-02,3199.52,1829.57,766.81,1222.17,294.31,1499.79,7272.28,1023.96,17108.41
2020-03,1887.05,1353.18,881.82,6241.21,0.00,1929.08,486.51,802.96,13581.81
2020-04,3821.84,1938.30,1603.18,0.00,2222.60,388.92,3195.04,3063.17,16233.05
2020-05,0.00,2501.62,2893.98,3822.55,318.05,1318.17,4288.25,1703.51,16846.13


In [101]:
monthly["prev_month_spending"] = monthly["total_spending"].shift(1)

In [102]:
monthly["avg_spending_last_3_months"] = (
    monthly["total_spending"].rolling(3).mean()
)

In [103]:
monthly["spending_growth_rate"] = (
    monthly["total_spending"].pct_change()
)

In [104]:
monthly["month_number"] = monthly.index.month

In [105]:
monthly["next_month_spending"] = monthly["total_spending"].shift(-1)

In [106]:
monthly.head()

Category,Entertainment,Food & Drink,Health & Fitness,Rent,Salary,Shopping,Travel,Utilities,total_spending,prev_month_spending,avg_spending_last_3_months,spending_growth_rate,month_number,next_month_spending
Month,,,,,,,,,,,,,,
2020-01,1914.85,5458.51,2370.34,3663.22,1077.09,1178.66,0.00,1475.58,17138.25,NaN,NaN,NaN,1,17108.41
2020-02,3199.52,1829.57,766.81,1222.17,294.31,1499.79,7272.28,1023.96,17108.41,17138.25,NaN,-0.001741,2,13581.81
2020-03,1887.05,1353.18,881.82,6241.21,0.00,1929.08,486.51,802.96,13581.81,17108.41,15942.823333,-0.206133,3,16233.05
2020-04,3821.84,1938.30,1603.18,0.00,2222.60,388.92,3195.04,3063.17,16233.05,13581.81,15641.090000,0.195205,4,16846.13
2020-05,0.00,2501.62,2893.98,3822.55,318.05,1318.17,4288.25,1703.51,16846.13,16233.05,15553.663333,0.037767,5,20680.72


In [107]:
monthly.tail()

Category,Entertainment,Food & Drink,Health & Fitness,Rent,Salary,Shopping,Travel,Utilities,total_spending,prev_month_spending,avg_spending_last_3_months,spending_growth_rate,month_number,next_month_spending
Month,,,,,,,,,,,,,,
2024-08,5093.52,4312.83,1187.60,1839.16,3792.15,4232.91,2475.52,0.00,22933.69,27716.08,24401.280000,-0.172549,8,19383.94
2024-09,2674.54,3049.01,2724.02,1771.18,1879.91,2755.08,3492.63,1037.57,19383.94,22933.69,23344.570000,-0.154783,9,22984.21
2024-10,1157.94,1241.60,4453.31,1072.64,4178.35,2232.86,4452.56,4194.95,22984.21,19383.94,21767.280000,0.185735,10,23565.96
2024-11,1611.37,1441.13,1215.55,2946.19,5653.77,1847.25,7874.49,976.21,23565.96,22984.21,21978.036667,0.025311,11,10938.92
2024-12,727.25,406.37,1428.01,4601.56,784.35,2830.56,0.00,160.82,10938.92,23565.96,19163.030000,-0.535817,12,NaN


In [108]:
monthly = monthly.dropna()

In [109]:
monthly.head()

Category,Entertainment,Food & Drink,Health & Fitness,Rent,Salary,Shopping,Travel,Utilities,total_spending,prev_month_spending,avg_spending_last_3_months,spending_growth_rate,month_number,next_month_spending
Month,,,,,,,,,,,,,,
2020-03,1887.05,1353.18,881.82,6241.21,0.00,1929.08,486.51,802.96,13581.81,17108.41,15942.823333,-0.206133,3,16233.05
2020-04,3821.84,1938.30,1603.18,0.00,2222.60,388.92,3195.04,3063.17,16233.05,13581.81,15641.090000,0.195205,4,16846.13
2020-05,0.00,2501.62,2893.98,3822.55,318.05,1318.17,4288.25,1703.51,16846.13,16233.05,15553.663333,0.037767,5,20680.72
2020-06,4092.44,426.30,4901.31,1547.80,1107.84,2611.18,990.97,5002.88,20680.72,16846.13,17919.966667,0.227624,6,24026.27
2020-07,4853.64,1786.39,3031.75,3919.73,109.93,2158.36,5355.49,2810.98,24026.27,20680.72,20517.706667,0.161771,7,31386.80


In [110]:
monthly.isna().sum()

Category
Entertainment                 0
Food & Drink                  0
Health & Fitness              0
Rent                          0
Salary                        0
Shopping                      0
Travel                        0
Utilities                     0
total_spending                0
prev_month_spending           0
avg_spending_last_3_months    0
spending_growth_rate          0
month_number                  0
next_month_spending           0
dtype: int64

In [111]:
X = monthly.drop("next_month_spending", axis=1)

In [112]:
y = monthly["next_month_spending"]

In [34]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

In [113]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [114]:
model = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [115]:
predictions = model.predict(X_test)

In [116]:
mae = mean_absolute_error(y_test, predictions)
print("MAE:", mae)

MAE: 3490.5140083333304


In [117]:
r2 = r2_score(y_test, predictions)
print("R2:", r2)

R2: -0.1845406753576191


In [119]:
print(monthly.shape)

(57, 14)


In [120]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

r2 = r2_score(y_test, predictions)
mae = mean_absolute_error(y_test, predictions)
mse = mean_squared_error(y_test, predictions)
rmse = np.sqrt(mse)

print("R2 Score:", r2)
print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)

R2 Score: -0.1845406753576191
MAE: 3490.5140083333304
MSE: 19199532.525673915
RMSE: 4381.727116751329


In [121]:
import joblib

joblib.dump(model, "expense_model.pkl")

print("Model saved as expense_model.pkl")

Model saved as expense_model.pkl


In [122]:
joblib.dump(X.columns.tolist(), "expense_model_features.pkl")

['expense_model_features.pkl']